In [1]:
import requests
import pandas as pd
import time
from datetime import datetime

# -----------------------------
# SETTINGS
# -----------------------------
SUBREDDITS = ["CBD", "cannabis", "trees", "weed", "Marijuana"]

KEYWORDS = [
    "CBD",
    "edible",
    "gummies",
    "vape",
    "oil",
    "tincture",
    "topical"
]

START_DATE = "2022-01-01"
END_DATE = "2025-01-01"

POSTS_OUTPUT_FILE = "reddit_posts.csv"
COMMENTS_OUTPUT_FILE = "reddit_comments.csv"

REQUEST_SLEEP_SECONDS = 1

# -----------------------------
# HELPERS
# -----------------------------
def monthly_ranges(start_date, end_date):
    start = pd.Timestamp(start_date, tz="UTC")
    end = pd.Timestamp(end_date, tz="UTC")
    current = start

    while current < end:
        next_month = current + pd.offsets.MonthBegin(1)
        if next_month > end:
            next_month = end
        yield int(current.timestamp()), int(next_month.timestamp())
        current = next_month


def clean_text(text):
    if pd.isna(text):
        return ""
    return str(text).replace("\n", " ").replace("\r", " ").strip()


def fetch_pullpush(endpoint, subreddit, keyword, after_ts, before_ts, size=100):
    url = f"https://api.pullpush.io/reddit/search/{endpoint}/"
    params = {
        "subreddit": subreddit,
        "q": keyword,
        "after": after_ts,
        "before": before_ts,
        "size": size,
        "sort": "asc",
        "sort_type": "created_utc"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    return data.get("data", [])


def collect_data(endpoint_name):
    """
    endpoint_name should be either:
    - 'submission' for posts
    - 'comment' for comments
    """
    all_rows = []

    print(f"\nStarting download for: {endpoint_name}")

    for subreddit in SUBREDDITS:
        for keyword in KEYWORDS:
            print(f"\nSearching {endpoint_name}: subreddit='{subreddit}', keyword='{keyword}'")

            for after_ts, before_ts in monthly_ranges(START_DATE, END_DATE):
                start_str = datetime.utcfromtimestamp(after_ts).strftime("%Y-%m-%d")
                end_str = datetime.utcfromtimestamp(before_ts).strftime("%Y-%m-%d")

                try:
                    rows = fetch_pullpush(
                        endpoint=endpoint_name,
                        subreddit=subreddit,
                        keyword=keyword,
                        after_ts=after_ts,
                        before_ts=before_ts,
                        size=100
                    )

                    print(f"  {start_str} to {end_str}: {len(rows)} rows")

                    for row in rows:
                        row["searched_subreddit"] = subreddit
                        row["searched_keyword"] = keyword
                        row["source_type"] = endpoint_name

                    all_rows.extend(rows)
                    time.sleep(REQUEST_SLEEP_SECONDS)

                except Exception as e:
                    print(f"  ERROR for {endpoint_name} | {subreddit} | {keyword} | {start_str} to {end_str}")
                    print(f"  {e}")
                    time.sleep(2)

    print(f"\nFinished {endpoint_name}. Raw rows collected: {len(all_rows)}")
    return pd.DataFrame(all_rows)


def prepare_posts_df(df):
    if df.empty:
        return df

    wanted_cols = [
        "id",
        "subreddit",
        "title",
        "selftext",
        "author",
        "created_utc",
        "score",
        "num_comments",
        "url",
        "searched_subreddit",
        "searched_keyword",
        "source_type"
    ]
    existing_cols = [c for c in wanted_cols if c in df.columns]
    df = df[existing_cols].copy()

    if "title" in df.columns:
        df["title"] = df["title"].apply(clean_text)

    if "selftext" in df.columns:
        df["selftext"] = df["selftext"].apply(clean_text)

    if "created_utc" in df.columns:
        df["date_utc"] = pd.to_datetime(df["created_utc"], unit="s", utc=True)
        df["year"] = df["date_utc"].dt.year
        df["month"] = df["date_utc"].dt.month

    if "title" in df.columns and "selftext" in df.columns:
        df["full_text"] = (df["title"].fillna("") + " " + df["selftext"].fillna("")).str.strip()

    if "id" in df.columns:
        before = len(df)
        df = df.drop_duplicates(subset="id")
        print(f"Posts: removed {before - len(df)} duplicate rows.")

    if "full_text" in df.columns:
        df = df[df["full_text"].str.strip() != ""]

    return df


def prepare_comments_df(df):
    if df.empty:
        return df

    wanted_cols = [
        "id",
        "subreddit",
        "author",
        "body",
        "created_utc",
        "score",
        "link_id",
        "parent_id",
        "searched_subreddit",
        "searched_keyword",
        "source_type"
    ]
    existing_cols = [c for c in wanted_cols if c in df.columns]
    df = df[existing_cols].copy()

    if "body" in df.columns:
        df["body"] = df["body"].apply(clean_text)

    if "created_utc" in df.columns:
        df["date_utc"] = pd.to_datetime(df["created_utc"], unit="s", utc=True)
        df["year"] = df["date_utc"].dt.year
        df["month"] = df["date_utc"].dt.month

    if "id" in df.columns:
        before = len(df)
        df = df.drop_duplicates(subset="id")
        print(f"Comments: removed {before - len(df)} duplicate rows.")

    if "body" in df.columns:
        df = df[df["body"].str.strip() != ""]

    return df


def main():
    # Download posts
    posts_df_raw = collect_data("submission")
    posts_df = prepare_posts_df(posts_df_raw)

    # Download comments
    comments_df_raw = collect_data("comment")
    comments_df = prepare_comments_df(comments_df_raw)

    # Save
    posts_df.to_csv(POSTS_OUTPUT_FILE, index=False, encoding="utf-8-sig")
    comments_df.to_csv(COMMENTS_OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print("\n==========================")
    print("DOWNLOAD COMPLETE")
    print("==========================")
    print(f"Posts saved to: {POSTS_OUTPUT_FILE}")
    print(f"Comments saved to: {COMMENTS_OUTPUT_FILE}")
    print(f"Unique posts: {len(posts_df)}")
    print(f"Unique comments: {len(comments_df)}")


if __name__ == "__main__":
    main()


Starting download for: submission

Searching submission: subreddit='CBD', keyword='CBD'
  2022-01-01 to 2022-02-01: 100 rows
  2022-02-01 to 2022-03-01: 100 rows
  2022-03-01 to 2022-04-01: 100 rows
  2022-04-01 to 2022-05-01: 100 rows
  2022-05-01 to 2022-06-01: 100 rows
  2022-06-01 to 2022-07-01: 100 rows
  2022-07-01 to 2022-08-01: 100 rows
  2022-08-01 to 2022-09-01: 100 rows
  2022-09-01 to 2022-10-01: 100 rows
  2022-10-01 to 2022-11-01: 100 rows
  2022-11-01 to 2022-12-01: 100 rows
  2022-12-01 to 2023-01-01: 100 rows
  2023-01-01 to 2023-02-01: 100 rows
  2023-02-01 to 2023-03-01: 100 rows
  2023-03-01 to 2023-04-01: 100 rows
  2023-04-01 to 2023-05-01: 100 rows
  2023-05-01 to 2023-06-01: 100 rows
  2023-06-01 to 2023-07-01: 100 rows
  2023-07-01 to 2023-08-01: 100 rows
  2023-08-01 to 2023-09-01: 100 rows
  2023-09-01 to 2023-10-01: 100 rows
  2023-10-01 to 2023-11-01: 100 rows
  2023-11-01 to 2023-12-01: 100 rows
  2023-12-01 to 2024-01-01: 100 rows
  2024-01-01 to 2024-02